In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, re, json, math, time, random, shutil, zipfile, hashlib, gc, inspect, subprocess, unicodedata
import numpy as np
import pandas as pd
from collections import defaultdict

SEED = 42
TEXT_COL = "SN(Original Shona Tweet)"
LABEL_COL = "_label"
LABELS = ["NEG", "NEU", "POS"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

EXPECTED_ROWS = {"build": 7880, "validation": 876, "test": 1224, "lexicon": 6620}
EXPECTED_SHA256 = {
    "experimental_frozen_lexicon": "0cbfba6ea4c3e1bd84bc1fed31c364ab899e9c6a8e68314e10825bc070e74380",
    "multilingual_release": "29b12603ce20909cec029c2c9da83feb8263450e47b3ca0f7f2896bd1c66fc8f",
    "multilingual_release_core13": "b1137eebc77b9dc75042e8b00109198e55808a1f43d8b2e975a7badb25ea64d9",
    "build": "a3d9ca14798593ba8391af9a9a6ac9f5d55303ef174e5bd19d6dab96b9be8b24",
    "validation": "9a7a31c00cbb1bfdd369e6d06c1e68980b7faa5174884a9b88e70fec7c9f249c",
    "test": "055e52cded861127a566a93662f64ac5e8b65014437a79524afa3a4881ecc606",
}
MODEL_SPECS = {
    "AfroLM": "bonadossou/afrolm_active_learning",
    "AfroXLMR": "Davlan/afro-xlmr-base",
    "AfriBERTa": "castorini/afriberta_base",
}
# Pinned pretrained model revisions used in the source experiment.
MODEL_REVISIONS = {
    "AfroLM": "c726c59",
    "AfroXLMR": "25f2729",
    "AfriBERTa": "95b703f",
}
RECOMMENDED_GPU_SUBSTRING = "P100"
MAX_LENGTH = 128
EPOCHS = 4
EFFECTIVE_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
ALPHA_GRID = np.round(np.arange(0.0, 1.0001, 0.05), 2)
BOOTSTRAP_REPS = 2000
CLASSICAL_REPRODUCTION_TOLERANCE = 1e-6

RUN_XAI = True
LIME_NUM_SAMPLES = 500
LIME_NUM_FEATURES = 10

# Restrict the run to one GPU so the effective batch size remains identical on P100 and T4x2 sessions.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
INPUT_ROOT = Path(os.environ.get("FRENCHYSHONA_INPUT_ROOT", "/kaggle/input"))
WORKING_ROOT = Path(os.environ.get("FRENCHYSHONA_WORKING_ROOT", "/kaggle/working"))
OUTPUT_ROOT = WORKING_ROOT/"FrenchyShona_Part2_Clean_Shona_Models_Multilingual"
RESULTS_ZIP = WORKING_ROOT/"FrenchyShona_Part2_Clean_Shona_Models_Multilingual_Results.zip"
MODEL_ZIP_PATHS = {
    "AfroLM": WORKING_ROOT/"FrenchyShona_AfroLM_Model.zip",
    "AfroXLMR": WORKING_ROOT/"FrenchyShona_AfroXLMR_Model.zip",
    "AfriBERTa": WORKING_ROOT/"FrenchyShona_AfriBERTa_Model.zip",
}

for stale in [RESULTS_ZIP, *MODEL_ZIP_PATHS.values()]:
    if stale.exists():
        stale.unlink()

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
for p in [
    OUTPUT_ROOT,
    OUTPUT_ROOT/"config", OUTPUT_ROOT/"inputs", OUTPUT_ROOT/"splits",
    OUTPUT_ROOT/"tables", OUTPUT_ROOT/"predictions", OUTPUT_ROOT/"figures",
    OUTPUT_ROOT/"xai", OUTPUT_ROOT/"models", OUTPUT_ROOT/"logs",
]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

def ensure_package(import_name, pip_spec=None):
    try:
        return __import__(import_name)
    except ImportError:
        spec = pip_spec or import_name
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--quiet", spec],
                check=True,
            )
            return __import__(import_name)
        except Exception as exc:
            raise RuntimeError(
                f"Required package {import_name!r} could not be loaded. "
                "Turn Kaggle Internet on and Run All again."
            ) from exc

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_extract_zip(path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(path) as z:
        for member in z.infolist():
            target = (destination/member.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        z.extractall(destination)

def json_dump(path, obj):
    def clean(x):
        if isinstance(x, dict):
            return {str(k): clean(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return [clean(v) for v in x]
        if isinstance(x, np.generic):
            return clean(x.item())
        if isinstance(x, float) and not np.isfinite(x):
            return None
        if isinstance(x, Path):
            return str(x)
        return x
    path.write_text(
        json.dumps(clean(obj), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

print("Started UTC:", datetime.now(timezone.utc).isoformat())
print("Output:", OUTPUT_ROOT)

In [ ]:
# Kaggle has already extracted the two input ZIPs.
# Locate the authoritative datasets from their identifying files.

v74_markers = sorted(
    INPUT_ROOT.rglob("FrenchyShona_V2_v7_4_frozen.csv")
)

part1b_markers = sorted(
    INPUT_ROOT.rglob("FrenchyShona_V2_v7_4_multilingual_release.csv")
)

if len(v74_markers) != 1:
    raise RuntimeError(
        f"Expected exactly one V7.4 input, found {len(v74_markers)}: "
        f"{v74_markers}"
    )

if len(part1b_markers) != 1:
    raise RuntimeError(
        f"Expected exactly one Part 1B input, found {len(part1b_markers)}: "
        f"{part1b_markers}"
    )

v74_extract = v74_markers[0].parent
part1b_extract = part1b_markers[0].parent

print("V7.4 input:", v74_extract)
print("Part 1B input:", part1b_extract)


def one_file(root, name):
    matches = list(root.rglob(name))
    if len(matches) != 1:
        raise RuntimeError(f"Expected exactly one {name} under {root}; found {len(matches)}.")
    return matches[0]

v74_paths = {
    "experimental_frozen_lexicon": one_file(v74_extract, "FrenchyShona_V2_v7_4_frozen.csv"),
    "build": one_file(v74_extract, "train_build_for_lexicon.csv"),
    "validation": one_file(v74_extract, "train_validation_for_thresholds.csv"),
    "test": one_file(v74_extract, "test_strict_unseen.csv"),
    "validation_lexical_saved": one_file(v74_extract, "validation_lexicon_predictions.csv"),
    "test_lexical_saved": one_file(v74_extract, "strict_test_lexicon_predictions.csv"),
    "threshold_grid": one_file(v74_extract, "validation_lexical_threshold_grid.csv"),
    "lexicon_metrics_saved": one_file(v74_extract, "lexicon_only_metrics_strict_test.csv"),
    "classical_metrics_v7_4": one_file(v74_extract, "classical_baseline_metrics_strict_test.csv"),
}
part1b_paths = {
    "multilingual_release": one_file(part1b_extract, "FrenchyShona_V2_v7_4_multilingual_release.csv"),
    "multilingual_release_core13": one_file(part1b_extract, "FrenchyShona_V2_v7_4_multilingual_release_core13.csv"),
    "part2_gate": one_file(part1b_extract, "PART2_START_GATE.csv"),
    "public_release_gate": one_file(part1b_extract, "MULTILINGUAL_PUBLIC_RELEASE_GATE.csv"),
    "checksums": one_file(part1b_extract, "output_checksums.csv"),
}

# Verify the complete Part 1B output manifest, not only the release file.
part1b_root = part1b_paths["multilingual_release"].parent
checksum_table = pd.read_csv(part1b_paths["checksums"])
checksum_failures = []
for _, row in checksum_table.iterrows():
    path = part1b_root/str(row["relative_path"])
    if not path.exists():
        checksum_failures.append(f"missing:{row['relative_path']}")
    elif sha256_file(path) != str(row["sha256"]):
        checksum_failures.append(f"sha256:{row['relative_path']}")
    elif path.stat().st_size != int(row["size_bytes"]):
        checksum_failures.append(f"size:{row['relative_path']}")
if checksum_failures:
    raise RuntimeError("Part 1B output checksum verification failed: " + " | ".join(checksum_failures[:10]))

observed_hashes = {
    "experimental_frozen_lexicon": sha256_file(v74_paths["experimental_frozen_lexicon"]),
    "multilingual_release": sha256_file(part1b_paths["multilingual_release"]),
    "multilingual_release_core13": sha256_file(part1b_paths["multilingual_release_core13"]),
    "build": sha256_file(v74_paths["build"]),
    "validation": sha256_file(v74_paths["validation"]),
    "test": sha256_file(v74_paths["test"]),
}
for role, expected in EXPECTED_SHA256.items():
    if observed_hashes[role] != expected:
        raise RuntimeError(f"{role} SHA-256 mismatch. Expected {expected}, got {observed_hashes[role]}.")

part2_gate = pd.read_csv(part1b_paths["part2_gate"])
if len(part2_gate) != 1:
    raise AssertionError("PART2_START_GATE.csv must contain exactly one row.")

def bool_value(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().casefold() in {"true", "1", "yes", "pass"}

if not bool_value(part2_gate.iloc[0]["part2_start_allowed_after_result_review"]):
    raise RuntimeError("Part 1B has not cleared the Part 2 start gate.")

experimental_frozen_df = pd.read_csv(v74_paths["experimental_frozen_lexicon"])
lexicon_df = pd.read_csv(part1b_paths["multilingual_release"])
core13_df = pd.read_csv(part1b_paths["multilingual_release_core13"])
build_df = pd.read_csv(v74_paths["build"])
validation_df = pd.read_csv(v74_paths["validation"])
test_df = pd.read_csv(v74_paths["test"])
threshold_grid = pd.read_csv(v74_paths["threshold_grid"])
lexicon_metrics_v7_4 = pd.read_csv(v74_paths["lexicon_metrics_saved"])
classical_metrics_v7_4 = pd.read_csv(v74_paths["classical_metrics_v7_4"])
saved_validation_lex = pd.read_csv(v74_paths["validation_lexical_saved"])
saved_test_lex = pd.read_csv(v74_paths["test_lexical_saved"])

for name, frame in [("build", build_df), ("validation", validation_df), ("test", test_df)]:
    if len(frame) != EXPECTED_ROWS[name]:
        raise AssertionError(f"{name} rows: expected {EXPECTED_ROWS[name]}, got {len(frame)}.")
    for column in [TEXT_COL, LABEL_COL, "_norm_text"]:
        if column not in frame.columns:
            raise KeyError(f"{name} is missing required column {column!r}.")
    if set(frame[LABEL_COL].astype(str).str.upper()) != set(LABELS):
        raise AssertionError(f"{name} does not contain the expected three labels.")
    if not frame["_norm_text"].is_unique:
        raise AssertionError(f"{name} normalised text is not unique.")

if len(experimental_frozen_df) != 6620 or len(lexicon_df) != 6620 or len(core13_df) != 6620:
    raise AssertionError("The experimental frozen resource and both multilingual-release files must contain 6,620 rows.")

required_core13 = [
    "CILUBA", "French", "Score", "Sentiment", "Nature", "English",
    "Zulu", "Afrikaans", "Sepedi", "Xhosa", "Shona",
    "expanded_shona_class", "expanded_shona",
]
if list(core13_df.columns) != required_core13:
    raise AssertionError("The Part 1B core13 schema differs from the expected multilingual schema.")
if not core13_df.equals(lexicon_df[required_core13]):
    raise AssertionError("The core13 release does not match the corresponding columns in the audited release.")

scores = pd.to_numeric(lexicon_df["Score"], errors="coerce")
if scores.isna().any() or ((scores < -9) | (scores > 9)).any():
    raise AssertionError("The multilingual release contains missing, nonnumeric or out-of-range scores.")

# Independently verify that Part 1B only changed permitted multilingual metadata.
translation_fields = {"CILUBA", "French", "Nature", "English", "Zulu", "Afrikaans", "Sepedi", "Xhosa"}
protected_columns = [column for column in experimental_frozen_df.columns if column not in translation_fields]
missing_protected = [column for column in protected_columns if column not in lexicon_df.columns]
if missing_protected:
    raise AssertionError(f"Multilingual release is missing protected columns: {missing_protected}")
for column in protected_columns:
    left = experimental_frozen_df[column]
    right = lexicon_df[column]
    if column == "Score":
        equal = np.allclose(pd.to_numeric(left), pd.to_numeric(right), atol=0, rtol=0)
    else:
        equal = np.array_equal(
            left.astype("string").fillna("").to_numpy(),
            right.astype("string").fillna("").to_numpy(),
        )
    if not equal:
        raise AssertionError(f"Protected frozen column changed during Part 1B: {column}")

text_sets = {
    name: set(frame["_norm_text"])
    for name, frame in [("build", build_df), ("validation", validation_df), ("test", test_df)]
}
if text_sets["build"] & text_sets["validation"] or text_sets["build"] & text_sets["test"] or text_sets["validation"] & text_sets["test"]:
    raise AssertionError("Build, validation and strict test are not disjoint.")

for frame in [build_df, validation_df, test_df]:
    frame[TEXT_COL] = frame[TEXT_COL].astype(str)
    frame[LABEL_COL] = frame[LABEL_COL].astype(str).str.upper()
    frame["label_id"] = frame[LABEL_COL].map(LABEL2ID).astype(int)

required_threshold_columns = {
    "negative_threshold", "positive_threshold", "macro_f1", "accuracy",
    "neutral_width", "distance_from_zero",
}
if required_threshold_columns - set(threshold_grid.columns):
    raise KeyError("The V7.4 threshold grid is missing required columns.")
best_threshold = threshold_grid.sort_values(
    ["macro_f1", "accuracy", "distance_from_zero", "neutral_width", "negative_threshold", "positive_threshold"],
    ascending=[False, False, True, True, True, True],
    kind="mergesort",
).iloc[0]
NEG_THRESHOLD = float(best_threshold["negative_threshold"])
POS_THRESHOLD = float(best_threshold["positive_threshold"])
if (NEG_THRESHOLD, POS_THRESHOLD) != (2.5, 3.0):
    raise AssertionError(f"Unexpected V7.4 lexical thresholds: {(NEG_THRESHOLD, POS_THRESHOLD)}")

# Recompute source-language lexical evidence from the complete multilingual release.
def normalize_repeats(word, max_repeats=2):
    return re.sub(r"(.)\1{" + str(max_repeats) + r",}", lambda match: match.group(1) * max_repeats, str(word))

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).casefold()
    return re.sub(r"\s+", " ", text).strip()

def clean_lexical_text(value):
    text = normalize_text(value)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#([a-zA-Z]+)", r" \1 ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zA-Z'\-\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def tokenize_lexical_text(value):
    return [normalize_repeats(word, 2) for word in clean_lexical_text(value).split() if word]

def norm_token(value):
    return normalize_repeats(normalize_text(value).replace(" ", ""), 2)

source_score_lists = defaultdict(list)
for _, row in lexicon_df.iterrows():
    score = float(row["Score"])
    row_forms = set()
    for column in ["Shona", "expanded_shona"]:
        form_text = normalize_text(row.get(column, ""))
        if form_text and " " not in form_text:
            row_forms.add(norm_token(form_text))
    for form in row_forms:
        source_score_lists[form].append(score)
source_score_map = {word: float(np.median(values)) for word, values in source_score_lists.items()}

def tweet_features(text):
    tokens = tokenize_lexical_text(text)
    matched = [source_score_map[word] for word in tokens if word in source_score_map]
    if not matched:
        return {
            "match_count": 0, "token_count": len(tokens), "coverage_ratio": 0.0,
            "sum_score": 0.0, "avg_score": 0.0, "max_abs_score": 0.0, "score_std": 0.0,
        }
    values = np.asarray(matched, dtype=float)
    return {
        "match_count": len(matched), "token_count": len(tokens),
        "coverage_ratio": len(matched) / max(1, len(tokens)),
        "sum_score": float(values.sum()), "avg_score": float(values.mean()),
        "max_abs_score": float(np.max(np.abs(values))), "score_std": float(values.std()),
    }

def predict_from_average(frame):
    values = frame["avg_score"].to_numpy()
    matches = frame["match_count"].to_numpy()
    prediction = np.full(len(frame), "NEU", dtype=object)
    prediction[(matches > 0) & (values <= NEG_THRESHOLD)] = "NEG"
    prediction[(matches > 0) & (values >= POS_THRESHOLD)] = "POS"
    return prediction

def sign_rule(frame):
    values = frame["sum_score"].to_numpy()
    matches = frame["match_count"].to_numpy()
    prediction = np.full(len(frame), "NEU", dtype=object)
    prediction[(matches > 0) & (values < 0)] = "NEG"
    prediction[(matches > 0) & (values > 0)] = "POS"
    return prediction

def make_lexical_frame(source_frame):
    features = pd.DataFrame([tweet_features(text) for text in source_frame[TEXT_COL]])
    features["gold"] = source_frame[LABEL_COL].to_numpy()
    features["_norm_text"] = source_frame["_norm_text"].to_numpy()
    features["pred_calibrated"] = predict_from_average(features)
    return features

build_lex = make_lexical_frame(build_df)
validation_lex = make_lexical_frame(validation_df)
test_lex = make_lexical_frame(test_df)
test_lex["pred_raw"] = sign_rule(test_lex)

# Translation enrichment must not change any source Shona lexical feature.
def compare_saved_lexical(recomputed, saved, split_name):
    if not np.array_equal(recomputed["_norm_text"].astype(str), saved["_norm_text"].astype(str)):
        raise AssertionError(f"{split_name} lexical text alignment changed.")
    checks = {
        "match_count_equal": np.array_equal(recomputed["match_count"].to_numpy(), saved["match_count"].to_numpy()),
        "token_count_equal": np.array_equal(recomputed["token_count"].to_numpy(), saved["token_count"].to_numpy()),
        "avg_score_equal": np.allclose(recomputed["avg_score"].to_numpy(), saved["avg_score"].to_numpy(), atol=1e-12, rtol=0),
        "sum_score_equal": np.allclose(recomputed["sum_score"].to_numpy(), saved["sum_score"].to_numpy(), atol=1e-12, rtol=0),
        "prediction_equal": np.array_equal(recomputed["pred_calibrated"].astype(str), saved["pred_calibrated"].astype(str)),
    }
    if not all(checks.values()):
        raise AssertionError(f"{split_name} source lexical features changed after multilingual enrichment: {checks}")
    return {"split": split_name, **checks, "rows": len(recomputed)}

lexical_invariance = pd.DataFrame([
    compare_saved_lexical(validation_lex, saved_validation_lex, "validation"),
    compare_saved_lexical(test_lex, saved_test_lex, "strict_test"),
])
lexical_invariance.to_csv(OUTPUT_ROOT/"config"/"multilingual_release_source_lexical_invariance.csv", index=False)

from sklearn.metrics import accuracy_score, f1_score
lexicon_metrics_release = pd.DataFrame([
    {
        "model": "FrenchyShona V2 sign aggregate",
        "accuracy": accuracy_score(test_lex["gold"], test_lex["pred_raw"]),
        "macro_f1": f1_score(test_lex["gold"], test_lex["pred_raw"], labels=LABELS, average="macro", zero_division=0),
        "weighted_f1": f1_score(test_lex["gold"], test_lex["pred_raw"], labels=LABELS, average="weighted", zero_division=0),
    },
    {
        "model": "FrenchyShona V2 calibrated average",
        "accuracy": accuracy_score(test_lex["gold"], test_lex["pred_calibrated"]),
        "macro_f1": f1_score(test_lex["gold"], test_lex["pred_calibrated"], labels=LABELS, average="macro", zero_division=0),
        "weighted_f1": f1_score(test_lex["gold"], test_lex["pred_calibrated"], labels=LABELS, average="weighted", zero_division=0),
    },
])
lexicon_metrics_release["coverage"] = float((test_lex["match_count"] > 0).mean())
lexicon_metrics_release["strict_test_rows"] = len(test_lex)
lexicon_metrics_release["validation_selected_neg_threshold"] = NEG_THRESHOLD
lexicon_metrics_release["validation_selected_pos_threshold"] = POS_THRESHOLD
lexicon_metrics_release.to_csv(OUTPUT_ROOT/"tables"/"lexicon_only_metrics_multilingual_release.csv", index=False)

metric_compare = lexicon_metrics_v7_4.merge(
    lexicon_metrics_release,
    on="model",
    suffixes=("_v7_4", "_multilingual_release"),
)
for metric in ["accuracy", "macro_f1", "weighted_f1", "coverage"]:
    metric_compare[f"abs_diff_{metric}"] = (
        metric_compare[f"{metric}_v7_4"] - metric_compare[f"{metric}_multilingual_release"]
    ).abs()
if metric_compare[[f"abs_diff_{metric}" for metric in ["accuracy", "macro_f1", "weighted_f1", "coverage"]]].to_numpy().max() > 1e-12:
    raise AssertionError("Multilingual release changed the frozen Shona lexicon-only results.")
metric_compare.to_csv(OUTPUT_ROOT/"config"/"multilingual_release_lexicon_metric_invariance.csv", index=False)

# Save exact inputs inside the Part 2 result package.
for path in [*v74_paths.values(), *part1b_paths.values()]:
    destination = OUTPUT_ROOT/"inputs"/path.name
    if not destination.exists():
        shutil.copy2(path, destination)
for name, frame in [("build", build_df), ("validation", validation_df), ("test", test_df)]:
    frame.to_csv(OUTPUT_ROOT/"splits"/f"{name}.csv", index=False)
validation_lex.to_csv(OUTPUT_ROOT/"inputs"/"validation_lexicon_predictions_from_multilingual_release.csv", index=False)
test_lex.to_csv(OUTPUT_ROOT/"inputs"/"strict_test_lexicon_predictions_from_multilingual_release.csv", index=False)

input_manifest = pd.DataFrame([
    {"role": role, "file": str(path), "rows": len(pd.read_csv(path)), "sha256": sha256_file(path)}
    for role, path in {**v74_paths, **part1b_paths}.items()
    if path.suffix.casefold() == ".csv"
])
input_manifest.to_csv(OUTPUT_ROOT/"config"/"input_manifest.csv", index=False)

MULTILINGUAL_FIELDS_FOR_TRANSFER = [
    "CILUBA", "French", "English", "Zulu", "Afrikaans", "Sepedi", "Xhosa", "Shona", "expanded_shona"
]
run_config = {
    "part": 2,
    "notebook_version": "part2-multilingual-v2",
    "seed": SEED,
    "class_order": LABELS,
    "rows": EXPECTED_ROWS,
    "input_sha256": observed_hashes,
    "part1b_start_gate_pass": True,
    "lexical_thresholds_from_v7_4_validation": {
        "negative_threshold": NEG_THRESHOLD,
        "positive_threshold": POS_THRESHOLD,
    },
    "model_specs": MODEL_SPECS,
    "model_revisions": MODEL_REVISIONS,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "hybrid_alpha_grid": ALPHA_GRID.tolist(),
    "hybrid_primary_rule": "coverage-aware; no lexical match leaves contextual probabilities unchanged",
    "source_lexical_fields": ["Shona", "expanded_shona"],
    "multilingual_fields_preserved_for_transfer": MULTILINGUAL_FIELDS_FOR_TRANSFER,
    "recommended_kaggle_accelerator": "GPU P100",
}
json_dump(OUTPUT_ROOT/"config"/"run_config.json", run_config)

print("Input validation: PASS")
print("Build / validation / strict test:", len(build_df), len(validation_df), len(test_df))
print("Build class counts:", build_df[LABEL_COL].value_counts().to_dict())
print("Validation class counts:", validation_df[LABEL_COL].value_counts().to_dict())
print("Strict-test class counts:", test_df[LABEL_COL].value_counts().to_dict())
print("Canonical multilingual lexicon rows:", len(lexicon_df))
print("Multilingual release SHA-256:", observed_hashes["multilingual_release"])
print("V7.4 lexical thresholds:", NEG_THRESHOLD, POS_THRESHOLD)
print("Source lexical invariance after translation enrichment: PASS")


In [ ]:
# Majority and clean TF-IDF classical baselines.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    f1_score, confusion_matrix, roc_auc_score,
)
from sklearn.preprocessing import label_binarize

def softmax_np(values):
    values = np.asarray(values, dtype=float)
    values = values - np.max(values, axis=1, keepdims=True)
    exp_values = np.exp(values)
    return exp_values / exp_values.sum(axis=1, keepdims=True)

def metric_row(
    name, y_true, y_pred, scores=None,
    train_seconds=np.nan, inference_seconds=np.nan,
    result_source="part2_clean_rerun",
):
    pm, rm, fm, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    pw, rw, fw, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    row = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": pm,
        "macro_recall": rm,
        "macro_f1": fm,
        "weighted_precision": pw,
        "weighted_recall": rw,
        "weighted_f1": fw,
        "train_seconds": train_seconds,
        "strict_test_inference_seconds": inference_seconds,
        "result_source": result_source,
    }
    if scores is not None:
        try:
            row["macro_ovr_roc_auc"] = roc_auc_score(
                label_binarize(y_true, classes=[0, 1, 2]),
                scores,
                average="macro",
                multi_class="ovr",
            )
        except Exception:
            row["macro_ovr_roc_auc"] = np.nan
    else:
        row["macro_ovr_roc_auc"] = np.nan
    return row

y_build = build_df["label_id"].to_numpy()
y_val = validation_df["label_id"].to_numpy()
y_test = test_df["label_id"].to_numpy()

results = []
for _, source_row in lexicon_metrics_release.iterrows():
    results.append({
        "model": str(source_row["model"]),
        "accuracy": float(source_row["accuracy"]),
        "balanced_accuracy": np.nan,
        "macro_precision": np.nan,
        "macro_recall": np.nan,
        "macro_f1": float(source_row["macro_f1"]),
        "weighted_precision": np.nan,
        "weighted_recall": np.nan,
        "weighted_f1": float(source_row["weighted_f1"]),
        "train_seconds": 0.0,
        "strict_test_inference_seconds": np.nan,
        "macro_ovr_roc_auc": np.nan,
        "result_source": "part1b_multilingual_release_recomputed_lexicon_evaluation",
    })

majority_id = int(pd.Series(y_build).mode().iloc[0])
majority_pred = np.full(len(test_df), majority_id)
results.append(metric_row("Majority baseline", y_test, majority_pred))

classical_specs = {
    "TF-IDF Linear SVM": LinearSVC(
        class_weight="balanced", random_state=SEED
    ),
    "TF-IDF Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=SEED
    ),
    "TF-IDF Multinomial Naive Bayes": MultinomialNB(),
}

for model_name, classifier in classical_specs.items():
    pipeline = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
            ),
        ),
        ("classifier", classifier),
    ])
    started = time.perf_counter()
    pipeline.fit(build_df[TEXT_COL], y_build)
    train_seconds = time.perf_counter() - started

    started = time.perf_counter()
    prediction = pipeline.predict(test_df[TEXT_COL])
    inference_seconds = time.perf_counter() - started

    if hasattr(pipeline, "predict_proba"):
        score = pipeline.predict_proba(test_df[TEXT_COL])
    else:
        score = softmax_np(pipeline.decision_function(test_df[TEXT_COL]))

    results.append(metric_row(
        model_name,
        y_test,
        prediction,
        score,
        train_seconds=train_seconds,
        inference_seconds=inference_seconds,
    ))

    output = test_df[["_norm_text", TEXT_COL, LABEL_COL, "label_id"]].copy()
    output["pred_id"] = prediction
    output["pred_label"] = [ID2LABEL[i] for i in prediction]
    for class_id, label in enumerate(LABELS):
        output[f"prob_{label}"] = score[:, class_id]
    output.to_csv(
        OUTPUT_ROOT/"predictions"/(
            re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_") + ".csv"
        ),
        index=False,
    )

new_classical = pd.DataFrame(results)
new_classical = new_classical[new_classical["model"].isin(classical_specs)].copy()
comparison = classical_metrics_v7_4.merge(
    new_classical[["model", "accuracy", "macro_f1", "weighted_f1"]],
    on="model",
    how="outer",
    suffixes=("_v7_4", "_part2"),
    indicator=True,
)
for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    comparison[f"abs_diff_{metric}"] = (
        comparison[f"{metric}_v7_4"] - comparison[f"{metric}_part2"]
    ).abs()
comparison.to_csv(
    OUTPUT_ROOT/"tables"/"classical_v7_4_reproduction_audit.csv",
    index=False,
)
if not comparison["_merge"].eq("both").all():
    raise AssertionError("Classical model names did not align with V7.4.")
maximum_difference = comparison[
    [f"abs_diff_{metric}" for metric in ["accuracy", "macro_f1", "weighted_f1"]]
].to_numpy(dtype=float).max()
if maximum_difference > CLASSICAL_REPRODUCTION_TOLERANCE:
    raise AssertionError(
        f"Classical rerun differs from V7.4 by {maximum_difference:.3g}. "
        "Inspect classical_v7_4_reproduction_audit.csv."
    )

pd.DataFrame(results).to_csv(
    OUTPUT_ROOT/"tables"/"source_model_metrics_running.csv",
    index=False,
)
print(pd.DataFrame(results)[
    ["model", "accuracy", "macro_f1", "weighted_f1", "macro_ovr_roc_auc"]
])

In [ ]:
# Transformer training utilities.

torch = ensure_package("torch")
ensure_package("transformers", "transformers>=4.44,<5")
ensure_package("sentencepiece")
ensure_package("accelerate", "accelerate>=0.34,<2")

from torch.utils.data import Dataset
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)

set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=y_build,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

def load_tokenizer(source, revision=None):
    errors = []
    for use_fast in (True, False):
        try:
            return AutoTokenizer.from_pretrained(source, revision=revision, use_fast=use_fast)
        except Exception as exc:
            errors.append(
                f"use_fast={use_fast}: {type(exc).__name__}: {exc}"
            )
    raise RuntimeError(
        f"Tokenizer could not be loaded from {source}. " + " | ".join(errors)
    )

class TextDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.texts = frame[TEXT_COL].astype(str).tolist()
        self.labels = frame["label_id"].astype(int).tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        item = self.tokenizer(
            self.texts[index],
            truncation=True,
            max_length=MAX_LENGTH,
        )
        item["labels"] = self.labels[index]
        return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_trainer_metrics(eval_prediction):
    logits, labels = eval_prediction
    prediction = np.argmax(logits, axis=1)
    row = metric_row(
        "temporary", labels, prediction, softmax_np(logits)
    )
    return {
        key: row[key]
        for key in [
            "accuracy", "balanced_accuracy",
            "macro_precision", "macro_recall", "macro_f1",
            "weighted_precision", "weighted_recall", "weighted_f1",
            "macro_ovr_roc_auc",
        ]
    }

def make_training_args(model_name, device_batch):
    checkpoint_dir = OUTPUT_ROOT/"models"/f"{model_name}_checkpoints"
    parameters = inspect.signature(TrainingArguments.__init__).parameters
    arguments = dict(
        output_dir=str(checkpoint_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=device_batch,
        per_device_eval_batch_size=device_batch,
        gradient_accumulation_steps=max(
            1, EFFECTIVE_BATCH_SIZE // device_batch
        ),
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        data_seed=SEED,
        fp16=bool(torch.cuda.is_available()),
    )
    arguments[
        "eval_strategy" if "eval_strategy" in parameters
        else "evaluation_strategy"
    ] = "epoch"
    arguments["save_strategy"] = "epoch"
    arguments["logging_strategy"] = "steps"
    return TrainingArguments(**arguments)

def save_prediction_frame(frame, probabilities, model_name, split_name):
    prediction = np.argmax(probabilities, axis=1)
    output = frame[["_norm_text", TEXT_COL, LABEL_COL, "label_id"]].copy()
    output["pred_id"] = prediction
    output["pred_label"] = [ID2LABEL[i] for i in prediction]
    for class_id, label in enumerate(LABELS):
        output[f"prob_{label}"] = probabilities[:, class_id]
    path = OUTPUT_ROOT/"predictions"/f"{model_name.lower()}_{split_name}.csv"
    output.to_csv(path, index=False)
    return output

def train_one_model(model_name, model_id):
    model_revision = MODEL_REVISIONS[model_name]
    last_error = None
    for device_batch in [16, 8, 4]:
        try:
            print(f"\nTraining {model_name} with device batch {device_batch}")
            tokenizer = load_tokenizer(model_id, revision=model_revision)
            model = AutoModelForSequenceClassification.from_pretrained(
                model_id,
                revision=model_revision,
                num_labels=3,
                id2label=ID2LABEL,
                label2id=LABEL2ID,
                ignore_mismatched_sizes=True,
            )
            datasets = {
                "build": TextDataset(build_df, tokenizer),
                "validation": TextDataset(validation_df, tokenizer),
                "test": TextDataset(test_df, tokenizer),
            }
            trainer = WeightedTrainer(
                model=model,
                args=make_training_args(model_name, device_batch),
                train_dataset=datasets["build"],
                eval_dataset=datasets["validation"],
                data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                compute_metrics=compute_trainer_metrics,
                class_weights=class_weights_tensor,
            )

            started = time.perf_counter()
            trainer.train()
            train_seconds = time.perf_counter() - started

            model_dir = OUTPUT_ROOT/"models"/model_name
            trainer.save_model(model_dir)
            tokenizer.save_pretrained(model_dir)

            started = time.perf_counter()
            validation_logits = trainer.predict(
                datasets["validation"]
            ).predictions
            validation_seconds = time.perf_counter() - started

            started = time.perf_counter()
            test_logits = trainer.predict(datasets["test"]).predictions
            test_seconds = time.perf_counter() - started

            validation_probabilities = softmax_np(validation_logits)
            test_probabilities = softmax_np(test_logits)
            save_prediction_frame(
                validation_df,
                validation_probabilities,
                model_name,
                "validation",
            )
            save_prediction_frame(
                test_df,
                test_probabilities,
                model_name,
                "strict_test",
            )

            selected_checkpoint = str(
                trainer.state.best_model_checkpoint or model_dir
            )
            test_result = metric_row(
                model_name,
                y_test,
                np.argmax(test_probabilities, axis=1),
                test_probabilities,
                train_seconds=train_seconds,
                inference_seconds=test_seconds,
            )
            test_result.update({
                "model_id": model_id,
                "model_revision": model_revision,
                "selected_checkpoint_during_training": selected_checkpoint,
                "final_saved_model_directory": str(model_dir),
                "best_validation_metric": trainer.state.best_metric,
                "device_batch_size": device_batch,
                "gradient_accumulation_steps": max(
                    1, EFFECTIVE_BATCH_SIZE // device_batch
                ),
                "validation_inference_seconds": validation_seconds,
            })

            pd.DataFrame(trainer.state.log_history).to_csv(
                OUTPUT_ROOT/"logs"/f"{model_name.lower()}_training_history.csv",
                index=False,
            )
            json_dump(
                OUTPUT_ROOT/"config"/f"{model_name.lower()}_manifest.json",
                test_result,
            )

            checkpoint_dir = OUTPUT_ROOT/"models"/f"{model_name}_checkpoints"
            if checkpoint_dir.exists():
                shutil.rmtree(checkpoint_dir)

            del trainer, model, tokenizer, datasets
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return test_result

        except RuntimeError as exc:
            last_error = exc
            if (
                "out of memory" not in str(exc).lower()
                or device_batch == 4
            ):
                raise
            print("CUDA OOM; retrying with a smaller device batch.")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    raise last_error

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if RECOMMENDED_GPU_SUBSTRING not in gpu_name.upper():
        print("WARNING: GPU P100 is the fixed recommended Kaggle accelerator; this run is using", gpu_name)
print("Build class weights:", dict(zip(LABELS, class_weights)))

In [ ]:
# Fine-tune the three fixed African-focused PLMs.

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a Kaggle GPU accelerator before running Part 2."
    )

for model_name, model_id in MODEL_SPECS.items():
    result = train_one_model(model_name, model_id)
    results.append(result)
    pd.DataFrame(results).to_csv(
        OUTPUT_ROOT/"tables"/"source_model_metrics_running.csv",
        index=False,
    )

print(pd.DataFrame(results)[
    ["model", "accuracy", "macro_f1", "weighted_f1", "macro_ovr_roc_auc"]
])

In [ ]:
# Fixed ensembles, coverage-aware hybrids, ablations and paired bootstrap.

def load_probabilities(model_name, split_name):
    frame = pd.read_csv(
        OUTPUT_ROOT/"predictions"/f"{model_name.lower()}_{split_name}.csv"
    )
    expected = validation_df if split_name == "validation" else test_df
    if not np.array_equal(
        frame["_norm_text"].astype(str),
        expected["_norm_text"].astype(str),
    ):
        raise AssertionError(
            f"{model_name} {split_name} prediction alignment failed."
        )
    return frame[[f"prob_{label}" for label in LABELS]].to_numpy()

validation_probabilities = {
    name: load_probabilities(name, "validation")
    for name in MODEL_SPECS
}
test_probabilities = {
    name: load_probabilities(name, "strict_test")
    for name in MODEL_SPECS
}

def lexical_onehot(series):
    ids = pd.Series(series).map(LABEL2ID).astype(int).to_numpy()
    output = np.zeros((len(ids), 3), dtype=float)
    output[np.arange(len(ids)), ids] = 1.0
    return ids, output

validation_lexical_ids, validation_lexical_onehot = lexical_onehot(
    validation_lex["pred_calibrated"]
)
test_lexical_ids, test_lexical_onehot = lexical_onehot(
    test_lex["pred_calibrated"]
)
validation_match = validation_lex["match_count"].to_numpy() > 0
test_match = test_lex["match_count"].to_numpy() > 0

def select_alpha(base_probabilities):
    rows = []
    for alpha in ALPHA_GRID:
        hybrid = base_probabilities.copy()
        hybrid[validation_match] = (
            alpha * base_probabilities[validation_match]
            + (1 - alpha) * validation_lexical_onehot[validation_match]
        )
        prediction = np.argmax(hybrid, axis=1)
        rows.append({
            "alpha": float(alpha),
            "validation_macro_f1": f1_score(
                y_val, prediction, average="macro", zero_division=0
            ),
            "validation_accuracy": accuracy_score(y_val, prediction),
        })
    grid = pd.DataFrame(rows)
    # Conservative deterministic tie-break: preserve more contextual probability.
    best = grid.sort_values(
        ["validation_macro_f1", "validation_accuracy", "alpha"],
        ascending=[False, False, False],
        kind="mergesort",
    ).iloc[0]
    return grid, best

def paired_bootstrap(
    y_true, prediction_a, prediction_b,
    reps=BOOTSTRAP_REPS, seed=SEED,
):
    rng = np.random.default_rng(seed)
    observed = (
        f1_score(
            y_true, prediction_b, average="macro", zero_division=0
        )
        - f1_score(
            y_true, prediction_a, average="macro", zero_division=0
        )
    )
    differences = []
    n = len(y_true)
    for _ in range(reps):
        indices = rng.integers(0, n, n)
        differences.append(
            f1_score(
                y_true[indices],
                prediction_b[indices],
                average="macro",
                zero_division=0,
            )
            - f1_score(
                y_true[indices],
                prediction_a[indices],
                average="macro",
                zero_division=0,
            )
        )
    low, high = np.quantile(differences, [0.025, 0.975])
    return {
        "observed_delta_macro_f1": observed,
        "ci_low": low,
        "ci_high": high,
        "excludes_zero": bool(low > 0 or high < 0),
        "replicates": reps,
    }

system_validation = dict(validation_probabilities)
system_test = dict(test_probabilities)

ensemble_members = {
    "Ensemble_AfroXLMR_AfriBERTa": ["AfroXLMR", "AfriBERTa"],
    "Ensemble_AfroLM_AfroXLMR_AfriBERTa": [
        "AfroLM", "AfroXLMR", "AfriBERTa"
    ],
}
for ensemble_name, members in ensemble_members.items():
    system_validation[ensemble_name] = np.mean(
        [validation_probabilities[member] for member in members],
        axis=0,
    )
    system_test[ensemble_name] = np.mean(
        [test_probabilities[member] for member in members],
        axis=0,
    )

validation_predictions = validation_df[
    ["_norm_text", TEXT_COL, LABEL_COL, "label_id"]
].copy()
strict_predictions = test_df[
    ["_norm_text", TEXT_COL, LABEL_COL, "label_id"]
].copy()

strict_predictions["lexicon_match_count"] = test_lex["match_count"].to_numpy()
strict_predictions["lexicon_pred_label"] = test_lex[
    "pred_calibrated"
].to_numpy()
validation_predictions["lexicon_match_count"] = validation_lex[
    "match_count"
].to_numpy()
validation_predictions["lexicon_pred_label"] = validation_lex[
    "pred_calibrated"
].to_numpy()

alpha_grids = []
bootstrap_rows = []

for system_name in system_test:
    validation_base = system_validation[system_name]
    test_base = system_test[system_name]
    validation_base_prediction = np.argmax(validation_base, axis=1)
    test_base_prediction = np.argmax(test_base, axis=1)

    if system_name.startswith("Ensemble"):
        results.append(metric_row(
            system_name.replace("_", " "),
            y_test,
            test_base_prediction,
            test_base,
        ))

    for class_id, label in enumerate(LABELS):
        validation_predictions[
            f"{system_name}_prob_{label}"
        ] = validation_base[:, class_id]
        strict_predictions[
            f"{system_name}_prob_{label}"
        ] = test_base[:, class_id]
    validation_predictions[
        f"{system_name}_pred"
    ] = validation_base_prediction
    strict_predictions[
        f"{system_name}_pred"
    ] = test_base_prediction

    grid, best = select_alpha(validation_base)
    grid["base_system"] = system_name
    alpha_grids.append(grid)
    alpha = float(best["alpha"])

    validation_hybrid = validation_base.copy()
    validation_hybrid[validation_match] = (
        alpha * validation_base[validation_match]
        + (1 - alpha) * validation_lexical_onehot[validation_match]
    )

    test_hybrid = test_base.copy()
    test_hybrid[test_match] = (
        alpha * test_base[test_match]
        + (1 - alpha) * test_lexical_onehot[test_match]
    )
    test_hybrid_prediction = np.argmax(test_hybrid, axis=1)

    hybrid_name = (
        "Coverage-aware hybrid "
        + system_name.replace("_", " ")
        + " + FrenchyShona"
    )
    row = metric_row(
        hybrid_name,
        y_test,
        test_hybrid_prediction,
        test_hybrid,
    )
    row.update({
        "selected_alpha": alpha,
        "validation_macro_f1": float(best["validation_macro_f1"]),
        "lexical_coverage": float(test_match.mean()),
        "hybrid_rule": "coverage_aware",
    })
    results.append(row)

    validation_predictions[
        f"hybrid_{system_name}_pred"
    ] = np.argmax(validation_hybrid, axis=1)
    strict_predictions[
        f"hybrid_{system_name}_pred"
    ] = test_hybrid_prediction
    for class_id, label in enumerate(LABELS):
        validation_predictions[
            f"hybrid_{system_name}_prob_{label}"
        ] = validation_hybrid[:, class_id]
        strict_predictions[
            f"hybrid_{system_name}_prob_{label}"
        ] = test_hybrid[:, class_id]

    bootstrap = paired_bootstrap(
        y_test, test_base_prediction, test_hybrid_prediction
    )
    bootstrap.update({
        "base_system": system_name,
        "hybrid_system": hybrid_name,
        "selected_alpha": alpha,
    })
    bootstrap_rows.append(bootstrap)

    # Historical always-active sensitivity analysis; not the primary system.
    always_active = (
        alpha * test_base
        + (1 - alpha) * test_lexical_onehot
    )
    always_prediction = np.argmax(always_active, axis=1)
    ablation_name = (
        "Always-active ablation "
        + system_name.replace("_", " ")
        + " + FrenchyShona"
    )
    ablation_row = metric_row(
        ablation_name,
        y_test,
        always_prediction,
        always_active,
    )
    ablation_row.update({
        "selected_alpha": alpha,
        "validation_macro_f1": np.nan,
        "lexical_coverage": float(test_match.mean()),
        "hybrid_rule": "always_active_ablation",
    })
    results.append(ablation_row)

pd.concat(alpha_grids, ignore_index=True).to_csv(
    OUTPUT_ROOT/"tables"/"source_hybrid_validation_alpha_grid.csv",
    index=False,
)
pd.DataFrame(bootstrap_rows).to_csv(
    OUTPUT_ROOT/"tables"/"source_paired_bootstrap_hybrid_vs_contextual.csv",
    index=False,
)
validation_predictions.to_csv(
    OUTPUT_ROOT/"predictions"/"all_source_system_predictions_validation.csv",
    index=False,
)
strict_predictions.to_csv(
    OUTPUT_ROOT/"predictions"/"all_source_system_predictions_strict_test.csv",
    index=False,
)

results_df = (
    pd.DataFrame(results)
    .drop_duplicates("model", keep="last")
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
results_df.to_csv(
    OUTPUT_ROOT/"tables"/"source_model_metrics_final.csv",
    index=False,
)

three_model_name = "Ensemble_AfroLM_AfroXLMR_AfriBERTa"
all_alpha = pd.concat(alpha_grids, ignore_index=True)
three_best = all_alpha[
    all_alpha["base_system"].eq(three_model_name)
].sort_values(
    ["validation_macro_f1", "validation_accuracy", "alpha"],
    ascending=[False, False, False],
).iloc[0]

source_freeze = {
    "class_order": LABELS,
    "model_specs": MODEL_SPECS,
    "ensemble_two": ["AfroXLMR", "AfriBERTa"],
    "ensemble_three": ["AfroLM", "AfroXLMR", "AfriBERTa"],
    "primary_hybrid_base": three_model_name,
    "primary_hybrid_alpha": float(three_best["alpha"]),
    "primary_hybrid_rule": "coverage_aware",
    "lexical_negative_threshold": NEG_THRESHOLD,
    "lexical_positive_threshold": POS_THRESHOLD,
    "frozen_lexicon_sha256": EXPECTED_SHA256["multilingual_release"],
    "canonical_multilingual_release_sha256": EXPECTED_SHA256["multilingual_release"],
    "canonical_multilingual_release_core13_sha256": EXPECTED_SHA256["multilingual_release_core13"],
    "experimental_v7_4_frozen_lexicon_sha256": EXPECTED_SHA256["experimental_frozen_lexicon"],
    "source_lexical_fields": ["Shona", "expanded_shona"],
    "transfer_multilingual_fields": MULTILINGUAL_FIELDS_FOR_TRANSFER,
    "provisional_ciluba_policy": "exclude provisional Part 1B Ciluba forms from the primary transfer inventory; include them only in a named sensitivity analysis",
    "build_sha256": EXPECTED_SHA256["build"],
    "validation_sha256": EXPECTED_SHA256["validation"],
    "strict_test_sha256": EXPECTED_SHA256["test"],
}
json_dump(
    OUTPUT_ROOT/"config"/"part2_source_system_freeze.json",
    source_freeze,
)

print(results_df[[
    "model", "accuracy", "macro_f1", "weighted_f1",
    "macro_ovr_roc_auc",
]].head(20))
print("Primary three-model hybrid alpha:", source_freeze["primary_hybrid_alpha"])

In [ ]:
# Source-task figures.

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve

def safe_name(value):
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")

plot_frame = results_df.head(18).sort_values("macro_f1")
plt.figure(figsize=(10, 8))
plt.barh(plot_frame["model"], plot_frame["macro_f1"])
plt.xlabel("Macro F1")
plt.title("Clean Shona Source-Task Model Comparison")
plt.tight_layout()
plt.savefig(
    OUTPUT_ROOT/"figures"/"fig_source_model_comparison_macro_f1.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()

principal_predictions = {
    "TF-IDF Linear SVM": pd.read_csv(
        OUTPUT_ROOT/"predictions"/"tf_idf_linear_svm.csv"
    ),
    "TF-IDF Logistic Regression": pd.read_csv(
        OUTPUT_ROOT/"predictions"/"tf_idf_logistic_regression.csv"
    ),
    "TF-IDF Multinomial Naive Bayes": pd.read_csv(
        OUTPUT_ROOT/"predictions"/"tf_idf_multinomial_naive_bayes.csv"
    ),
    "AfroLM": pd.read_csv(
        OUTPUT_ROOT/"predictions"/"afrolm_strict_test.csv"
    ),
    "AfroXLMR": pd.read_csv(
        OUTPUT_ROOT/"predictions"/"afroxlmr_strict_test.csv"
    ),
    "AfriBERTa": pd.read_csv(
        OUTPUT_ROOT/"predictions"/"afriberta_strict_test.csv"
    ),
}

for name, frame in principal_predictions.items():
    prediction = frame["pred_id"].to_numpy()
    matrix = confusion_matrix(y_test, prediction, labels=[0, 1, 2])
    display = ConfusionMatrixDisplay(matrix, display_labels=LABELS)
    figure, axis = plt.subplots(figsize=(5.5, 4.8))
    display.plot(ax=axis, values_format="d", colorbar=False)
    axis.set_title(name)
    figure.tight_layout()
    figure.savefig(
        OUTPUT_ROOT/"figures"/f"confusion_{safe_name(name)}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(figure)

three_prediction = strict_predictions[
    "Ensemble_AfroLM_AfroXLMR_AfriBERTa_pred"
].to_numpy()
hybrid_prediction = strict_predictions[
    "hybrid_Ensemble_AfroLM_AfroXLMR_AfriBERTa_pred"
].to_numpy()

for name, prediction in [
    ("Three-model ensemble", three_prediction),
    ("Coverage-aware three-model FrenchyShona hybrid", hybrid_prediction),
]:
    matrix = confusion_matrix(y_test, prediction, labels=[0, 1, 2])
    display = ConfusionMatrixDisplay(matrix, display_labels=LABELS)
    figure, axis = plt.subplots(figsize=(5.5, 4.8))
    display.plot(ax=axis, values_format="d", colorbar=False)
    axis.set_title(name)
    figure.tight_layout()
    figure.savefig(
        OUTPUT_ROOT/"figures"/f"confusion_{safe_name(name)}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(figure)

y_binary = label_binarize(y_test, classes=[0, 1, 2])

plt.figure(figsize=(7.5, 6))
for name in [
    "TF-IDF Linear SVM",
    "TF-IDF Logistic Regression",
    "TF-IDF Multinomial Naive Bayes",
]:
    frame = principal_predictions[name]
    scores = frame[[f"prob_{label}" for label in LABELS]].to_numpy()
    grid = np.linspace(0, 1, 1000)
    mean_tpr = np.zeros_like(grid)
    for class_id in range(3):
        fpr, tpr, _ = roc_curve(
            y_binary[:, class_id], scores[:, class_id]
        )
        mean_tpr += np.interp(grid, fpr, tpr) / 3
    model_auc = roc_auc_score(
        y_binary, scores, average="macro", multi_class="ovr"
    )
    plt.plot(
        grid, mean_tpr,
        label=f"{name} (macro AUC={model_auc:.3f})",
    )
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Classical Models — Macro One-vs-Rest ROC")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    OUTPUT_ROOT/"figures"/"fig_source_classical_macro_ovr_roc.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()

roc_systems = {
    "AfroLM": np.column_stack([
        strict_predictions[f"AfroLM_prob_{label}"] for label in LABELS
    ]),
    "AfroXLMR": np.column_stack([
        strict_predictions[f"AfroXLMR_prob_{label}"] for label in LABELS
    ]),
    "AfriBERTa": np.column_stack([
        strict_predictions[f"AfriBERTa_prob_{label}"] for label in LABELS
    ]),
    "Three-model ensemble": np.column_stack([
        strict_predictions[
            f"Ensemble_AfroLM_AfroXLMR_AfriBERTa_prob_{label}"
        ]
        for label in LABELS
    ]),
    "Coverage-aware hybrid": np.column_stack([
        strict_predictions[
            f"hybrid_Ensemble_AfroLM_AfroXLMR_AfriBERTa_prob_{label}"
        ]
        for label in LABELS
    ]),
}

plt.figure(figsize=(7.5, 6))
for name, scores in roc_systems.items():
    grid = np.linspace(0, 1, 1000)
    mean_tpr = np.zeros_like(grid)
    for class_id in range(3):
        fpr, tpr, _ = roc_curve(
            y_binary[:, class_id], scores[:, class_id]
        )
        mean_tpr += np.interp(grid, fpr, tpr) / 3
    model_auc = roc_auc_score(
        y_binary, scores, average="macro", multi_class="ovr"
    )
    plt.plot(
        grid, mean_tpr,
        label=f"{name} (macro AUC={model_auc:.3f})",
    )
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("PLMs, Ensemble and Hybrid — Macro One-vs-Rest ROC")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(
    OUTPUT_ROOT/"figures"/"fig_source_plm_ensemble_hybrid_macro_ovr_roc.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()

print("Saved source-task figures.")

In [ ]:
# Post-hoc attention and LIME diagnostics.
# XAI failures are logged and do not invalidate the quantitative model run.

if RUN_XAI:
    xai_status = []
    try:
        ensure_package("lime")
        from lime.lime_text import LimeTextExplainer
        from transformers import (
            AutoTokenizer, AutoModelForSequenceClassification
        )

        best_alpha = source_freeze["primary_hybrid_alpha"]
        test_work = test_df[
            ["_norm_text", TEXT_COL, LABEL_COL, "label_id"]
        ].copy()
        test_work["afriberta_pred"] = strict_predictions[
            "AfriBERTa_pred"
        ].to_numpy()
        test_work["hybrid_pred"] = hybrid_prediction
        test_work["ensemble_pred"] = three_prediction
        test_work["hybrid_correct"] = test_work[
            "hybrid_pred"
        ].eq(test_work["label_id"])
        test_work["afriberta_correct"] = test_work[
            "afriberta_pred"
        ].eq(test_work["label_id"])
        test_work["hybrid_helped"] = (
            ~test_work["afriberta_correct"]
            & test_work["hybrid_correct"]
        )
        test_work["hybrid_hurt"] = (
            test_work["afriberta_correct"]
            & ~test_work["hybrid_correct"]
        )

        selections = []
        criteria = [
            ("correct_NEG", test_work["hybrid_correct"] & test_work["label_id"].eq(0)),
            ("correct_NEU", test_work["hybrid_correct"] & test_work["label_id"].eq(1)),
            ("correct_POS", test_work["hybrid_correct"] & test_work["label_id"].eq(2)),
            ("hybrid_misclassified", ~test_work["hybrid_correct"]),
            ("hybrid_helped", test_work["hybrid_helped"]),
            ("hybrid_hurt", test_work["hybrid_hurt"]),
        ]
        used = set()
        for example_type, mask in criteria:
            subset = test_work[
                mask & ~test_work["_norm_text"].isin(used)
            ]
            if len(subset):
                row = subset.iloc[0].copy()
                row["example_type"] = example_type
                selections.append(row)
                used.add(row["_norm_text"])
        xai_examples = pd.DataFrame(selections)
        xai_examples.to_csv(
            OUTPUT_ROOT/"xai"/"xai_selected_examples.csv",
            index=False,
        )

        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        def predict_from_model_dir(model_name, texts, batch_size=32):
            model_dir = OUTPUT_ROOT/"models"/model_name
            tokenizer = load_tokenizer(model_dir)
            model = AutoModelForSequenceClassification.from_pretrained(
                model_dir
            ).to(device)
            model.eval()
            probability_batches = []
            for start in range(0, len(texts), batch_size):
                encoded = tokenizer(
                    list(texts[start:start+batch_size]),
                    padding=True,
                    truncation=True,
                    max_length=MAX_LENGTH,
                    return_tensors="pt",
                )
                encoded = {
                    key: value.to(device)
                    for key, value in encoded.items()
                }
                with torch.no_grad():
                    probabilities = torch.softmax(
                        model(**encoded).logits,
                        dim=1,
                    ).cpu().numpy()
                probability_batches.append(probabilities)
            del model, tokenizer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return np.vstack(probability_batches)

        # Reuse the exact source-language matcher built from the canonical multilingual release.
        score_map = source_score_map

        def lexical_details(text):
            tokens = tokenize_lexical_text(text)
            matched = [
                (token, score_map[token])
                for token in tokens if token in score_map
            ]
            if not matched:
                return "NEU", 0.0, matched
            average = float(np.mean([score for _, score in matched]))
            prediction = (
                "NEG" if average <= NEG_THRESHOLD
                else "POS" if average >= POS_THRESHOLD
                else "NEU"
            )
            return prediction, average, matched

        def lexical_onehot_for_texts(texts):
            vectors = []
            counts = []
            for text in texts:
                prediction, _, matched = lexical_details(text)
                vector = np.zeros(3)
                vector[LABEL2ID[prediction]] = 1.0
                vectors.append(vector)
                counts.append(len(matched))
            return np.vstack(vectors), np.asarray(counts)

        match_rows = []
        for example_index, row in xai_examples.iterrows():
            prediction, average, matched = lexical_details(row[TEXT_COL])
            if matched:
                for token, score in matched:
                    match_rows.append({
                        "example_index": example_index,
                        "example_type": row["example_type"],
                        "token": token,
                        "score": score,
                        "average_lexical_score": average,
                        "lexical_prediction": prediction,
                    })
            else:
                match_rows.append({
                    "example_index": example_index,
                    "example_type": row["example_type"],
                    "token": "",
                    "score": np.nan,
                    "average_lexical_score": 0.0,
                    "lexical_prediction": "NEU",
                })
        pd.DataFrame(match_rows).to_csv(
            OUTPUT_ROOT/"xai"/"xai_frenchyshona_token_matches.csv",
            index=False,
        )

        def afriberta_function(texts):
            return predict_from_model_dir(
                "AfriBERTa", list(texts)
            )

        def hybrid_function(texts):
            probabilities = np.mean([
                predict_from_model_dir("AfroLM", list(texts)),
                predict_from_model_dir("AfroXLMR", list(texts)),
                predict_from_model_dir("AfriBERTa", list(texts)),
            ], axis=0)
            lexical_vectors, match_counts = lexical_onehot_for_texts(texts)
            output = probabilities.copy()
            match_mask = match_counts > 0
            output[match_mask] = (
                best_alpha * probabilities[match_mask]
                + (1 - best_alpha) * lexical_vectors[match_mask]
            )
            return output

        explainer = LimeTextExplainer(
            class_names=LABELS,
            random_state=SEED,
            split_expression=r"\W+",
        )
        lime_rows = []
        lime_log = []
        for example_index, row in xai_examples.iterrows():
            text = str(row[TEXT_COL])
            for model_name, function in [
                ("AfriBERTa", afriberta_function),
                ("Coverage-aware hybrid", hybrid_function),
            ]:
                try:
                    probabilities = function([text])[0]
                    prediction = int(np.argmax(probabilities))
                    explanation = explainer.explain_instance(
                        text,
                        function,
                        labels=[prediction],
                        num_features=LIME_NUM_FEATURES,
                        num_samples=LIME_NUM_SAMPLES,
                    )
                    weights = explanation.as_list(label=prediction)
                    for token, weight in weights:
                        lime_rows.append({
                            "example_index": example_index,
                            "example_type": row["example_type"],
                            "model": model_name,
                            "gold_label": ID2LABEL[int(row["label_id"])],
                            "predicted_label": ID2LABEL[prediction],
                            "token": token,
                            "lime_weight": weight,
                        })
                    plot_items = list(reversed(weights))
                    plt.figure(figsize=(7, 4.5))
                    plt.barh(
                        [item[0] for item in plot_items],
                        [item[1] for item in plot_items],
                    )
                    plt.xlabel("LIME weight for predicted class")
                    plt.title(f"{model_name}: {row['example_type']}")
                    plt.tight_layout()
                    plt.savefig(
                        OUTPUT_ROOT/"figures"/(
                            f"lime_{safe_name(model_name)}_"
                            f"{safe_name(row['example_type'])}.png"
                        ),
                        dpi=300,
                        bbox_inches="tight",
                    )
                    plt.close()
                    lime_log.append({
                        "example_index": example_index,
                        "example_type": row["example_type"],
                        "model": model_name,
                        "status": "complete",
                    })
                except Exception as exc:
                    lime_log.append({
                        "example_index": example_index,
                        "example_type": row["example_type"],
                        "model": model_name,
                        "status": f"error:{type(exc).__name__}",
                        "detail": str(exc),
                    })
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
        pd.DataFrame(lime_rows).to_csv(
            OUTPUT_ROOT/"xai"/"lime_token_weights.csv",
            index=False,
        )
        pd.DataFrame(lime_log).to_csv(
            OUTPUT_ROOT/"xai"/"lime_generation_log.csv",
            index=False,
        )

        attention_log = []
        if len(xai_examples):
            attention_text = str(xai_examples.iloc[0][TEXT_COL])
            for model_name in MODEL_SPECS:
                model_dir = OUTPUT_ROOT/"models"/model_name
                try:
                    tokenizer = load_tokenizer(model_dir)
                    model = (
                        AutoModelForSequenceClassification
                        .from_pretrained(
                            model_dir,
                            output_attentions=True,
                        )
                        .to(device)
                    )
                    model.eval()
                    encoded = tokenizer(
                        attention_text,
                        truncation=True,
                        max_length=MAX_LENGTH,
                        return_tensors="pt",
                    )
                    tokens = tokenizer.convert_ids_to_tokens(
                        encoded["input_ids"][0]
                    )
                    encoded = {
                        key: value.to(device)
                        for key, value in encoded.items()
                    }
                    with torch.no_grad():
                        output = model(
                            **encoded,
                            output_attentions=True,
                        )
                    if not output.attentions:
                        raise RuntimeError(
                            "The model did not return attention tensors."
                        )
                    final_attention = (
                        output.attentions[-1][0]
                        .mean(dim=0)
                        .detach()
                        .cpu()
                        .numpy()
                    )
                    weights = final_attention[0]
                    plt.figure(
                        figsize=(max(8, len(tokens) * 0.35), 4.5)
                    )
                    plt.bar(range(len(tokens)), weights)
                    plt.xticks(
                        range(len(tokens)),
                        tokens,
                        rotation=70,
                        ha="right",
                        fontsize=8,
                    )
                    plt.ylabel("Attention weight")
                    plt.title(
                        f"{model_name}: final-layer mean-head attention"
                    )
                    plt.tight_layout()
                    plt.savefig(
                        OUTPUT_ROOT/"figures"/(
                            f"attention_{model_name.lower()}.png"
                        ),
                        dpi=300,
                        bbox_inches="tight",
                    )
                    plt.close()
                    attention_log.append({
                        "model": model_name,
                        "status": "complete",
                        "tokens": len(tokens),
                    })
                    del model, tokenizer
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                except Exception as exc:
                    attention_log.append({
                        "model": model_name,
                        "status": f"error:{type(exc).__name__}",
                        "detail": str(exc),
                    })
        pd.DataFrame(attention_log).to_csv(
            OUTPUT_ROOT/"xai"/"attention_generation_log.csv",
            index=False,
        )
        xai_status.append({"stage": "xai", "status": "completed_with_logs"})
    except Exception as exc:
        xai_status.append({
            "stage": "xai",
            "status": f"skipped_or_failed:{type(exc).__name__}",
            "detail": str(exc),
        })
        gc.collect()
        if "torch" in globals() and torch.cuda.is_available():
            torch.cuda.empty_cache()

    pd.DataFrame(xai_status).to_csv(
        OUTPUT_ROOT/"xai"/"xai_stage_status.csv",
        index=False,
    )

print("XAI stage finished; inspect xai_stage_status.csv and generation logs.")

In [ ]:
# Final manifests and archives.

model_manifest = []
for model_name in MODEL_SPECS:
    model_dir = OUTPUT_ROOT/"models"/model_name
    files = sorted(path for path in model_dir.rglob("*") if path.is_file())
    if not files or not (model_dir/"config.json").exists():
        raise RuntimeError(f"Final saved model directory is incomplete: {model_name}")
    model_manifest.append({
        "model": model_name,
        "model_id": MODEL_SPECS[model_name],
        "model_revision": MODEL_REVISIONS[model_name],
        "files": len(files),
        "total_bytes": sum(path.stat().st_size for path in files),
        "config_sha256": sha256_file(model_dir/"config.json"),
    })
pd.DataFrame(model_manifest).to_csv(
    OUTPUT_ROOT/"config"/"saved_model_manifest.csv",
    index=False,
)

records = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        records.append({
            "relative_path": str(path.relative_to(OUTPUT_ROOT)),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })
pd.DataFrame(records).to_csv(
    OUTPUT_ROOT/"output_manifest.csv",
    index=False,
)

results_stage = WORKING_ROOT/"FrenchyShona_Part2_Multilingual_Results_Stage"
if results_stage.exists():
    shutil.rmtree(results_stage)
results_stage.mkdir(parents=True)

for folder in [
    "config", "inputs", "splits", "tables",
    "predictions", "figures", "xai", "logs",
]:
    source = OUTPUT_ROOT/folder
    if source.exists():
        shutil.copytree(source, results_stage/folder)
shutil.copy2(
    OUTPUT_ROOT/"output_manifest.csv",
    results_stage/"output_manifest.csv",
)

with zipfile.ZipFile(
    RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in results_stage.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    Path("FrenchyShona_Part2_Multilingual_Results")
                    / path.relative_to(results_stage)
                ),
            )

model_zip_rows = []
for model_name in MODEL_SPECS:
    model_dir = OUTPUT_ROOT/"models"/model_name
    zip_path = MODEL_ZIP_PATHS[model_name]
    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        for path in model_dir.rglob("*"):
            if path.is_file():
                archive.write(
                    path,
                    arcname=str(
                        Path(model_name)
                        / path.relative_to(model_dir)
                    ),
                )
    with zipfile.ZipFile(zip_path) as archive:
        bad_member = archive.testzip()
        if bad_member:
            raise RuntimeError(
                f"Corrupt model archive at {bad_member}"
            )
    model_zip_rows.append({
        "model": model_name,
        "zip_path": str(zip_path),
        "size_bytes": zip_path.stat().st_size,
        "sha256": sha256_file(zip_path),
    })

pd.DataFrame(model_zip_rows).to_csv(
    WORKING_ROOT/"FrenchyShona_Part2_Model_Zips_Manifest.csv",
    index=False,
)

with zipfile.ZipFile(RESULTS_ZIP) as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise RuntimeError(
            f"Corrupt Part 2 results ZIP at {bad_member}"
        )

print("PART 2 COMPLETE")
print("Results:", RESULTS_ZIP)
for row in model_zip_rows:
    print(row["zip_path"])

In [ ]:
# Part 2 runtime and environment audit
# Records runtime and environment information without retraining the models.

from pathlib import Path
from datetime import datetime, timezone
import platform
import json
import zipfile
import shutil

import numpy as np
import pandas as pd
import torch
import transformers
import sklearn

AUDIT_DIR = WORKING_ROOT / "FrenchyShona_Part2_Runtime_Environment_Audit"
AUDIT_ZIP = WORKING_ROOT / "FrenchyShona_Part2_Runtime_Environment_Audit.zip"

if AUDIT_DIR.exists():
    shutil.rmtree(AUDIT_DIR)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

if AUDIT_ZIP.exists():
    AUDIT_ZIP.unlink()

# ---------------------------------------------------------
# 1. Preserve recorded model timings from the completed run
# ---------------------------------------------------------

metrics_path = OUTPUT_ROOT / "tables" / "source_model_metrics_final.csv"
metrics = pd.read_csv(metrics_path)

timing_columns = [
    "model",
    "train_seconds",
    "validation_inference_seconds",
    "strict_test_inference_seconds",
]

timing_summary = metrics[
    [c for c in timing_columns if c in metrics.columns]
].copy()

timing_summary["train_minutes"] = (
    timing_summary["train_seconds"] / 60.0
)

timing_summary.to_csv(
    AUDIT_DIR / "part2_model_runtime_summary.csv",
    index=False,
)

# ---------------------------------------------------------
# 2. Preserve Hugging Face runtime records
# ---------------------------------------------------------

hf_rows = []

for model_name in ["afrolm", "afroxlmr", "afriberta"]:
    log_path = OUTPUT_ROOT / "logs" / f"{model_name}_training_history.csv"
    history = pd.read_csv(log_path)

    train_rows = history[
        history.get("train_runtime", pd.Series(dtype=float)).notna()
    ]

    if len(train_rows):
        row = train_rows.iloc[-1]

        hf_rows.append({
            "model": model_name,
            "hf_train_runtime_seconds": row.get("train_runtime", np.nan),
            "train_samples_per_second": row.get(
                "train_samples_per_second", np.nan
            ),
            "train_steps_per_second": row.get(
                "train_steps_per_second", np.nan
            ),
            "total_flos": row.get("total_flos", np.nan),
            "train_loss": row.get("train_loss", np.nan),
        })

pd.DataFrame(hf_rows).to_csv(
    AUDIT_DIR / "part2_huggingface_training_runtime.csv",
    index=False,
)

# ---------------------------------------------------------
# 3. Capture the runtime environment
# ---------------------------------------------------------

environment = {
    "part": 2,
    "notebook_version": "part2-multilingual-v2",
    "original_run_started_utc":
        "2026-08-22T19:17:01.096893+00:00",
    "runtime_audit_captured_utc":
        datetime.now(timezone.utc).isoformat(),
    "kaggle_accelerator_setting":
        "T4 x2 allocation; notebook restricted to one visible GPU",
    "cuda_visible_devices":
        str(os.environ.get("CUDA_VISIBLE_DEVICES", "")),
    "cuda_available":
        bool(torch.cuda.is_available()),
    "visible_gpu_count":
        int(torch.cuda.device_count()),
    "actual_gpu":
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None,
    "python_version":
        platform.python_version(),
    "torch_version":
        torch.__version__,
    "torch_cuda_version":
        torch.version.cuda,
    "transformers_version":
        transformers.__version__,
    "sklearn_version":
        sklearn.__version__,
    "numpy_version":
        np.__version__,
    "pandas_version":
        pd.__version__,
    "seed":
        SEED,
    "build_rows":
        len(build_df),
    "validation_rows":
        len(validation_df),
    "strict_test_rows":
        len(test_df),
    "lexicon_rows":
        len(lexicon_df),
    "epochs":
        EPOCHS,
    "max_length":
        MAX_LENGTH,
    "effective_batch_size":
        EFFECTIVE_BATCH_SIZE,
    "learning_rate":
        LEARNING_RATE,
    "weight_decay":
        WEIGHT_DECAY,
}

with open(
    AUDIT_DIR / "part2_runtime_environment.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(environment, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------------
# 4. Timing interpretation note
# ---------------------------------------------------------

(AUDIT_DIR / "README_TIMING_SCOPE.txt").write_text(
    """FrenchyShona Part 2 Runtime Audit

train_seconds:
Wall-clock timing surrounding model fitting/fine-tuning in the
completed Part 2 run.

validation_inference_seconds:
Inference on the frozen 876-row validation partition for PLMs.

strict_test_inference_seconds:
Inference on the 1,224-row strict unseen test partition.

Ensembles and FrenchyShona hybrids reuse the saved component
probabilities. They do not involve additional model training.
Their probability-aggregation overhead was not treated as
independent model training time.

The successful Part 2 run used one visible Tesla T4 GPU.
The Kaggle T4 x2 allocation was deliberately restricted to
CUDA_VISIBLE_DEVICES=0 to preserve the fixed effective batch size.
""",
    encoding="utf-8",
)

# ---------------------------------------------------------
# 5. Package the audit
# ---------------------------------------------------------

with zipfile.ZipFile(
    AUDIT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in AUDIT_DIR.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    Path(AUDIT_DIR.name) / path.relative_to(AUDIT_DIR)
                ),
            )

print("RUNTIME AUDIT COMPLETE")
print(AUDIT_ZIP)
print()
print(timing_summary.to_string(index=False))
print()
print(json.dumps(environment, indent=2))